# 02 — Training Loop Manual: Klasifikasi FashionMNIST

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/riki-profile/ai-engineer-journey/blob/main/01-deep-learning/02_training_loop_manual.ipynb)

Di notebook sebelumnya kita melakukan gradient descent sepenuhnya manual. Sekarang kita memakai "perkakas" standar PyTorch, tapi **tetap menulis training loop sendiri** agar setiap langkahnya terlihat:

- `Dataset` & `DataLoader` untuk memuat data per *batch*.
- `nn.Module` buatan sendiri untuk mendefinisikan model.
- `nn.CrossEntropyLoss` sebagai loss klasifikasi.
- `torch.optim.Adam` sebagai optimizer.
- Loop: **forward → loss → backward → `optimizer.step()` → `optimizer.zero_grad()`**, plus loop validasi.

**Dataset**: [FashionMNIST](https://github.com/zalandoresearch/fashion-mnist), 70.000 gambar grayscale 28×28 dari 10 jenis pakaian (60.000 train, 10.000 test).

> 💡 Di Colab aktifkan GPU: **Runtime → Change runtime type → GPU**. Di CPU pun tetap bisa, hanya lebih lambat.

## 0. Setup

Import library, set seed, dan tentukan device.

In [ ]:
import os
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Subset, random_split
from torchvision import datasets, transforms

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("PyTorch:", torch.__version__, "| device:", device)

### Konfigurasi

Semua hyperparameter dikumpulkan di satu tempat agar mudah diubah.

`QUICK_RUN` dipakai untuk uji cepat (subset kecil data dan 1 epoch), misalnya di mesin tanpa GPU. Nilainya dibaca dari environment variable `QUICK_RUN=1`; secara default (misalnya di Colab) notebook memakai **dataset penuh**. Kamu juga bisa langsung menulis `QUICK_RUN = True`.

In [ ]:
QUICK_RUN = os.environ.get("QUICK_RUN") == "1"

BATCH_SIZE = 64
LEARNING_RATE = 1e-3
EPOCHS = 1 if QUICK_RUN else 10
VAL_FRACTION = 0.1   # 10% data train dipakai sebagai data validasi
DATA_DIR = "./data"

print("QUICK_RUN:", QUICK_RUN, "| EPOCHS:", EPOCHS)

## 1. Memuat Data

### 1.1 Transformasi

- `ToTensor()`: mengubah gambar PIL (nilai 0–255) menjadi tensor float `(1, 28, 28)` dengan nilai 0–1.
- `Normalize(mean, std)`: menggeser nilai agar rata-rata ≈ 0 dan std ≈ 1. Ini membantu training lebih stabil (ingat latihan normalisasi di notebook 01). Nilai 0.2860 dan 0.3530 adalah mean & std piksel data train FashionMNIST.

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.2860,), (0.3530,)),
])

# download=True: otomatis mengunduh dataset jika belum ada
full_train = datasets.FashionMNIST(DATA_DIR, train=True, download=True, transform=transform)
test_set = datasets.FashionMNIST(DATA_DIR, train=False, download=True, transform=transform)

CLASS_NAMES = full_train.classes
print("Jumlah train:", len(full_train), "| test:", len(test_set))
print("Kelas:", CLASS_NAMES)

### 1.2 Split train / validasi

Data **test** hanya dipakai sekali di akhir. Untuk memantau performa selama training, kita menyisihkan sebagian data train sebagai **validasi**.

Jika `QUICK_RUN` aktif, kita ambil subset kecil saja dengan `Subset`.

In [ ]:
n_val = int(len(full_train) * VAL_FRACTION)
n_train = len(full_train) - n_val
train_set, val_set = random_split(
    full_train, [n_train, n_val], generator=torch.Generator().manual_seed(42)
)

if QUICK_RUN:
    train_set = Subset(train_set, range(2000))
    val_set = Subset(val_set, range(500))
    test_set = Subset(test_set, range(500))

print(f"train: {len(train_set)} | val: {len(val_set)} | test: {len(test_set)}")

### 1.3 DataLoader

`DataLoader` membagi data menjadi *batch* dan (untuk train) mengacaknya setiap epoch dengan `shuffle=True`. Mengacak mencegah model menghafal urutan data. Data validasi/test tidak perlu diacak.

In [ ]:
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False)

# Intip satu batch untuk memastikan bentuknya benar
images, labels = next(iter(train_loader))
print("images:", images.shape, images.dtype)  # (batch, channel, tinggi, lebar)
print("labels:", labels.shape, labels[:10])

### 1.4 Visualisasi contoh gambar

Selalu lihat data sebelum melatih model. Karena gambar sudah dinormalisasi, kita kembalikan ke skala asli dulu (*denormalize*) sebelum ditampilkan.

In [ ]:
def denormalize(img):
    # Kebalikan dari Normalize: x_asli = x * std + mean
    return img * 0.3530 + 0.2860

fig, axes = plt.subplots(2, 8, figsize=(12, 3.5))
for ax, img, label in zip(axes.flat, images, labels):
    ax.imshow(denormalize(img).squeeze(), cmap="gray")
    ax.set_title(CLASS_NAMES[label], fontsize=8)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 2. Model: `nn.Module` Buatan Sendiri

Setiap model di PyTorch adalah turunan `nn.Module` dengan dua bagian:
- `__init__`: mendefinisikan *layer* (yang punya parameter/bobot).
- `forward`: mendefinisikan alur data dari input ke output.

Model kita adalah **MLP (Multi-Layer Perceptron)**:
`gambar 28×28 → flatten (784) → Linear 256 → ReLU → Dropout → Linear 128 → ReLU → Linear 10`

Output 10 angka disebut **logits** (skor mentah per kelas). Kita *tidak* memasang softmax di akhir karena `CrossEntropyLoss` sudah menghitung softmax secara internal.

In [ ]:
class FashionMLP(nn.Module):
    def __init__(self, in_features=28 * 28, hidden1=256, hidden2=128, n_classes=10, dropout=0.2):
        super().__init__()  # wajib dipanggil agar nn.Module terinisialisasi
        self.flatten = nn.Flatten()          # (N, 1, 28, 28) -> (N, 784)
        self.fc1 = nn.Linear(in_features, hidden1)
        self.fc2 = nn.Linear(hidden1, hidden2)
        self.fc3 = nn.Linear(hidden2, n_classes)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)   # mematikan neuron acak saat training (regularisasi)

    def forward(self, x):
        x = self.flatten(x)
        x = self.dropout(self.relu(self.fc1(x)))
        x = self.relu(self.fc2(x))
        return self.fc3(x)                   # logits, belum softmax


model = FashionMLP().to(device)  # pindahkan semua parameter ke device
print(model)

n_params = sum(p.numel() for p in model.parameters())
print(f"Jumlah parameter: {n_params:,}")

Uji cepat: masukkan satu batch dan pastikan bentuk output-nya `(batch, 10)`.

In [ ]:
with torch.no_grad():
    out = model(images.to(device))
print("output shape:", out.shape)
print("logits sampel pertama:", out[0].cpu().numpy().round(3))

## 3. Loss Function dan Optimizer

- **`nn.CrossEntropyLoss`**: loss standar klasifikasi multi-kelas. Input: logits `(N, 10)` dan label integer `(N,)`.
- **`torch.optim.Adam`**: optimizer yang menyesuaikan besar langkah per parameter. Ia menggantikan update manual `w -= lr * w.grad` dari notebook 01. `model.parameters()` memberi tahu optimizer parameter mana yang harus di-update.

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

## 4. Training Loop Manual

### 4.1 Fungsi training satu epoch

Untuk setiap batch:
1. **Forward**: `logits = model(x)`
2. **Loss**: `loss = loss_fn(logits, y)`
3. **Backward**: `loss.backward()` menghitung gradien semua parameter.
4. **Step**: `optimizer.step()` meng-update parameter memakai gradien.
5. **Zero grad**: `optimizer.zero_grad()` me-reset gradien (ingat: gradien terakumulasi!).

Urutan `zero_grad()` boleh di awal atau di akhir iterasi, yang penting gradien sudah bersih sebelum `backward()` berikutnya.

`model.train()` mengaktifkan mode training (misalnya Dropout aktif).

In [ ]:
def train_one_epoch(model, loader, loss_fn, optimizer):
    model.train()
    total_loss, total_benar, total_data = 0.0, 0, 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)

        logits = model(x)            # 1. forward
        loss = loss_fn(logits, y)    # 2. hitung loss
        loss.backward()              # 3. backward: hitung gradien
        optimizer.step()             # 4. update parameter
        optimizer.zero_grad()        # 5. reset gradien

        # Catat statistik. loss.item() adalah rata-rata per batch,
        # jadi dikali ukuran batch agar rata-rata akhirnya tepat.
        total_loss += loss.item() * x.size(0)
        total_benar += (logits.argmax(dim=1) == y).sum().item()
        total_data += x.size(0)

    return total_loss / total_data, total_benar / total_data

### 4.2 Fungsi evaluasi (validasi/test)

Perbedaannya dengan training:
- `model.eval()`: menonaktifkan Dropout (dan menyesuaikan BatchNorm jika ada).
- `torch.no_grad()`: tidak perlu gradien → lebih cepat dan hemat memori.
- **Tidak ada** `backward()` atau `optimizer.step()`; model hanya diukur, tidak diubah.

In [ ]:
@torch.no_grad()
def evaluate(model, loader, loss_fn):
    model.eval()
    total_loss, total_benar, total_data = 0.0, 0, 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss = loss_fn(logits, y)

        total_loss += loss.item() * x.size(0)
        total_benar += (logits.argmax(dim=1) == y).sum().item()
        total_data += x.size(0)

    return total_loss / total_data, total_benar / total_data

### 4.3 Menjalankan training

Setiap epoch: latih di seluruh data train, lalu ukur di data validasi. Riwayatnya disimpan untuk digrafikkan.

In [ ]:
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader, loss_fn, optimizer)
    val_loss, val_acc = evaluate(model, val_loader, loss_fn)

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    print(f"Epoch {epoch:2d}/{EPOCHS} | "
          f"train loss {train_loss:.4f} acc {train_acc:.3f} | "
          f"val loss {val_loss:.4f} acc {val_acc:.3f}")

## 5. Grafik Loss dan Akurasi

Cara membaca grafik:
- Train dan val loss **sama-sama turun** → model masih belajar dengan baik.
- Train loss terus turun tapi val loss **naik** → **overfitting** (model menghafal data train).

Di epoch awal, akurasi validasi sering **lebih tinggi** daripada train. Ini wajar: angka train adalah rata-rata selama epoch berjalan (saat model masih buruk di awal epoch) dan Dropout aktif saat training, sedangkan validasi diukur di akhir epoch tanpa Dropout.

(Jika `QUICK_RUN` aktif, hanya ada 1 epoch sehingga grafik hanya berisi satu titik.)

In [ ]:
epochs_range = range(1, EPOCHS + 1)
fig, ax = plt.subplots(1, 2, figsize=(11, 4))

ax[0].plot(epochs_range, history["train_loss"], "o-", label="train")
ax[0].plot(epochs_range, history["val_loss"], "o-", label="validasi")
ax[0].set_xlabel("epoch"); ax[0].set_ylabel("loss"); ax[0].set_title("Loss"); ax[0].legend()

ax[1].plot(epochs_range, history["train_acc"], "o-", label="train")
ax[1].plot(epochs_range, history["val_acc"], "o-", label="validasi")
ax[1].set_xlabel("epoch"); ax[1].set_ylabel("akurasi"); ax[1].set_title("Akurasi"); ax[1].legend()

for a in ax:
    a.set_xticks(list(epochs_range))  # tampilkan epoch sebagai bilangan bulat

plt.tight_layout()
plt.show()

## 6. Evaluasi Akhir di Data Test

Setelah selesai memilih hyperparameter berdasarkan data validasi, barulah kita mengukur performa di data test. Angka inilah yang dilaporkan sebagai performa model.

In [ ]:
test_loss, test_acc = evaluate(model, test_loader, loss_fn)
print(f"Test loss: {test_loss:.4f} | Test akurasi: {test_acc:.3f}")

## 7. Contoh Prediksi

Kita ambil beberapa gambar test dan menampilkan prediksi model. `softmax` mengubah logits menjadi probabilitas (jumlahnya 1) agar kita bisa melihat tingkat keyakinan model.

Judul **hijau** = prediksi benar, **merah** = salah.

In [ ]:
model.eval()
x_test, y_test = next(iter(test_loader))
with torch.no_grad():
    logits = model(x_test.to(device))
    probs = torch.softmax(logits, dim=1).cpu()
preds = probs.argmax(dim=1)

fig, axes = plt.subplots(3, 6, figsize=(12, 7))
for i, ax in enumerate(axes.flat):
    ax.imshow(denormalize(x_test[i]).squeeze(), cmap="gray")
    benar = preds[i] == y_test[i]
    ax.set_title(
        f"pred: {CLASS_NAMES[preds[i]]} ({probs[i, preds[i]]:.0%})\nasli: {CLASS_NAMES[y_test[i]]}",
        fontsize=8, color="green" if benar else "red",
    )
    ax.axis("off")
plt.tight_layout()
plt.show()

## Ringkasan

- `nn.Module`: definisikan layer di `__init__` dan alur data di `forward`.
- Training loop: **forward → loss → backward → `optimizer.step()` → `optimizer.zero_grad()`**.
- Validasi memakai `model.eval()` + `torch.no_grad()`, tanpa update bobot.
- Pantau train vs val loss untuk mendeteksi overfitting; data test dipakai sekali di akhir.
- Dengan 10 epoch penuh, MLP sederhana ini biasanya mencapai akurasi test sekitar **88–89%**. Di modul 02 (CNN) kita akan melampauinya.

## Latihan

Kerjakan sendiri di sel baru di bawah ini.

1. **Eksperimen arsitektur & optimizer.** Ubah model (misalnya tambah satu hidden layer, ubah ukuran hidden, atau hilangkan Dropout) dan coba ganti optimizer ke `torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9)`. Catat akurasi validasi setiap percobaan dalam tabel kecil, lalu simpulkan konfigurasi mana yang terbaik dan mengapa.
2. **Confusion matrix & kesalahan terbesar.** Hitung confusion matrix 10×10 di data test (cukup dengan PyTorch/NumPy, tanpa scikit-learn). Pasangan kelas mana yang paling sering tertukar? Tampilkan 8 gambar yang salah diprediksi dengan probabilitas tertinggi (model paling "yakin tapi salah").
3. **Early stopping & menyimpan model.** Modifikasi training loop agar menyimpan bobot terbaik berdasarkan val loss (`torch.save(model.state_dict(), "best_model.pt")`) dan berhenti jika val loss tidak membaik selama 3 epoch berturut-turut. Setelah itu, buat model baru, muat bobot dengan `load_state_dict`, lalu pastikan akurasi test-nya sama.